# FLUX.1-dev LoRA — Islamic Parametric Architecture

Production training notebook. Hyperparameters (locked to spec):

| Hyperparameter | Value |
|---|---|
| Base model | `black-forest-labs/FLUX.1-dev` |
| LoRA rank / alpha | 16 / 16 |
| Learning rate | 1e-4 |
| Resolution | 1024×1024 |
| Max steps | 800 (800–1000 allowed) |
| Optimizer | adamw8bit |
| Mixed precision | bf16 + gradient checkpointing |
| Trigger word | `in Islamic_Parametric style` |
| Output | `pytorch_lora_weights.safetensors` |

**Runtime:** Google Colab → Change runtime → GPU (T4 works slowly; A100/L4 recommended).
**Access:** FLUX.1-dev is gated — accept the license at huggingface.co/black-forest-labs/FLUX.1-dev and set `HF_TOKEN` below with a token that has access.

In [ ]:
import os
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    os.environ['HF_TOKEN'] = os.environ.get('HF_TOKEN', '')
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

In [ ]:
!pip install -q "diffusers>=0.31.0" "transformers>=4.49.0" "accelerate>=0.31.0" "safetensors>=0.4.0" "huggingface_hub>=0.23" tensorboard

In [ ]:
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import os
import subprocess
REPO_URL = 'https://github.com/shehab-hegab/flux-islamic-parametric'
if not os.path.exists('train_flux_lora.py'):
    try:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, 'repo'], check=True)
        os.chdir('repo')
    except Exception as exc:
        print('clone failed (upload repo files manually):', exc)
print('cwd ready:', os.getcwd())

In [ ]:
from pathlib import Path
ds = Path('dataset_islamic_parametric')
images = sorted(ds.glob('image_*.jpg')) if ds.exists() else []
captions = sorted((ds / 'captions').glob('*.txt')) if (ds / 'captions').exists() else []
print(f'images: {len(images)} captions: {len(captions)}')
assert len(images) >= 1, 'upload dataset_islamic_parametric/ first'
print((ds / 'captions' / 'image_01.txt').read_text() if captions else 'no captions yet')

In [ ]:
!python generate_captions.py --backend template || true
!python train_flux_lora.py --fetch-script

## Training launch
Runs `accelerate launch training/train_dreambooth_lora_flux.py` via the wrapper. On a Colab T4 expect roughly 2–4 hours for 800 steps at 1024² with batch 1 + grad accum 4 + bf16 + gradient checkpointing. A100 is substantially faster.

In [ ]:
!python train_flux_lora.py --dry-run

In [ ]:
%env TOKENIZERS_PARALLELISM=false
!python train_flux_lora.py --instance-dir dataset_islamic_parametric --output-dir output

In [ ]:
from pathlib import Path
weights = Path('output/pytorch_lora_weights.safetensors')
print('weights:', weights, 'exists:', weights.exists(), 'size_mb:', weights.stat().st_size // 1024**2 if weights.exists() else 0)

In [ ]:
!python inference_eval.py --check || true
!python inference_eval.py --lora-path output || true

In [ ]:
!python upload_to_hf.py --dry-run